In [19]:
import pandas as pd

# 1. CARREGUEM ELS FITXERS
ruta_renda = r"C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\raw\2022_renda_disponible_llars_per_persona.csv"
ruta_edificis = r"C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\raw\2025_Edificacions_any_con.csv"

print("📂 Carregant dades...")
df_renda = pd.read_csv(ruta_renda)
df_edificis = pd.read_csv(ruta_edificis)

# 2. NETEJA DE NOMS (Estandarditzem perquè coincideixin)
# Canviem els noms de l'excel d'Edificis perquè siguin iguals als de Renda
df_edificis = df_edificis.rename(columns={
    'Codi_districte': 'Codi_Districte',
    'Codi_barri': 'Codi_Barri',
    'Seccio_censal': 'Seccio_Censal'
})

# 3. CREACIÓ DE LA CLAU MESTRA (CODI ÚNIC)
# Per seguretat, creem el codi llarg tipus "0801901001"
# Això evita que la Secció 1 del Districte 1 es barregi amb la Secció 1 del Districte 2.

def generar_codi(df):
    # Convertim a text i afegim zeros a l'esquerra (padding)
    # Districte: 1 -> "01"
    # Secció: 5 -> "005"
    return (
        '08019' + 
        df['Codi_Districte'].astype(str).str.zfill(2) + 
        df['Seccio_Censal'].astype(str).str.zfill(3)
    )

df_renda['Codi_Unic'] = generar_codi(df_renda)
df_edificis['Codi_Unic'] = generar_codi(df_edificis)

print("🔑 Claus úniques generades (Exemple: 0801901001)")

# 4. EL MERGE (La Fusió)
# unim per 'Codi_Unic'.
# how='inner' -> Només es queda amb les zones que tenen DADES A TOTS DOS LLOCS.
df_master = pd.merge(
    df_renda[['Codi_Unic', 'Import_Euros', 'Nom_Districte', 'Nom_Barri']], # Agafem només el que ens interessa
    df_edificis[['Codi_Unic', 'Any_construccio', 'Nombre']], 
    on='Codi_Unic', 
    how='inner'
)

# 5. RESULTAT I GUARDAT
print(f"✅ FUSIÓ COMPLETADA! Tenim {df_master.shape[0]} zones censals llestes per la IA.")
print(df_master.head())

# Guardem el fitxer final a la carpeta processed
ruta_guardat = r"C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\processed\dataset_mestre.csv"

# Assegurem que la carpeta existeix
import os
os.makedirs(os.path.dirname(ruta_guardat), exist_ok=True)

df_master.to_csv(ruta_guardat, index=False)
print(f"💾 Arxiu guardat a: {ruta_guardat}")

📂 Carregant dades...
🔑 Claus úniques generades (Exemple: 0801901001)
✅ FUSIÓ COMPLETADA! Tenim 8331 zones censals llestes per la IA.
    Codi_Unic  Import_Euros Nom_Districte Nom_Barri Any_construccio  Nombre
0  0801901001         15940  Ciutat Vella  el Raval           <1901      31
1  0801901001         15940  Ciutat Vella  el Raval       1901-1940      10
2  0801901001         15940  Ciutat Vella  el Raval       1941-1950       9
3  0801901001         15940  Ciutat Vella  el Raval       1951-1960       4
4  0801901001         15940  Ciutat Vella  el Raval       1961-1970       4
💾 Arxiu guardat a: C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\processed\dataset_mestre.csv


In [25]:
import pandas as pd

# Carreguem sense tocar res
ruta_mestre = r"C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\processed\dataset_mestre.csv"
df = pd.read_csv(ruta_mestre)

print("--- EXEMPLE DE DADES BRUTES ---")
print("Renda (Import_Euros):", df['Import_Euros'].head(5).tolist())
print("Any (Any_construccio):", df['Any_construccio'].head(5).tolist())
print("Nombre d'Edificis:", df['Nombre'].head(5).tolist())

--- EXEMPLE DE DADES BRUTES ---
Renda (Import_Euros): [15940, 15940, 15940, 15940, 15940]
Any (Any_construccio): ['<1901', '1901-1940', '1941-1950', '1951-1960', '1961-1970']
Nombre d'Edificis: [31, 10, 9, 4, 4]


In [27]:
import pandas as pd
import numpy as np

# 1. CARREGUEM EL DATASET
ruta_mestre = r"C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\processed\dataset_mestre.csv"
df = pd.read_csv(ruta_mestre)

print(f"Original: {len(df)} files.")

# --- 🛠️ FASE DE TRADUCCIÓ D'ANYS ---
def convertir_rango_any(valor):
    valor = str(valor).strip() # Traiem espais
    
    # Mapeig manual dels intervals que hem vist
    if valor == '<1901': return 1900
    if valor == '1901-1940': return 1920
    if valor == '1941-1950': return 1945
    if valor == '1951-1960': return 1955
    if valor == '1961-1970': return 1965
    if valor == '1971-1980': return 1975
    if valor == '1981-1990': return 1985
    if valor == '1991-2000': return 1995
    if valor == '2001-2010': return 2005
    if valor == '2011-2020': return 2015
    if valor == '2021-2030': return 2025
    
    # Si troba un any sol (ex: "2022"), intenta convertir-lo directament
    try:
        return int(float(valor))
    except:
        return np.nan # Si no l'entén, el marca com a buit

print("🔄 Traduint intervals d'anys a números...")
df['Any_Calculat'] = df['Any_construccio'].apply(convertir_rango_any)

# Neteja extra per la columna Import i Nombre (per si de cas)
# Com que hem vist que Import ja és numèric [15940], no cal fer res especial,
# però ens assegurem que no hi hagi errors.
df['Import_Euros'] = pd.to_numeric(df['Import_Euros'], errors='coerce')
df['Nombre'] = pd.to_numeric(df['Nombre'], errors='coerce')

# Eliminem files que no haguem pogut traduir
df_net = df.dropna(subset=['Any_Calculat', 'Nombre', 'Import_Euros'])
print(f"Files vàlides per càlcul: {len(df_net)}")

# 2. CÀLCUL DE LA PONDERACIÓ
# Ara multipliquem l'any numèric (1920) pel nombre d'edificis (10)
df_net['Pes_Any'] = df_net['Any_Calculat'] * df_net['Nombre']

# 3. AGRUPACIÓ PER ZONA
df_agrupat = df_net.groupby('Codi_Unic').agg({
    'Pes_Any': 'sum',
    'Nombre': 'sum',            # Total edificis barri
    'Import_Euros': 'first',    # La renda es manté
    'Nom_Districte': 'first',
    'Nom_Barri': 'first'
}).reset_index()

# 4. RESULTAT FINAL (Mitjana Ponderada)
df_agrupat['Any_Mitja_Ponderat'] = df_agrupat['Pes_Any'] / df_agrupat['Nombre']
df_agrupat['Any_Mitja_Ponderat'] = df_agrupat['Any_Mitja_Ponderat'].astype(int)

# Seleccionem i guardem
df_final = df_agrupat[['Codi_Unic', 'Nom_Districte', 'Nom_Barri', 'Import_Euros', 'Any_Mitja_Ponderat', 'Nombre']]
df_final = df_final.rename(columns={'Nombre': 'Total_Edificis_Zona'})

print(f"✅ ÈXIT TOTAL! Tenim {len(df_final)} seccions censals úniques.")
print(df_final.head())

# Guardem
ruta_final = r"C:\Users\jordi\Documents\TFG_Projecte\TFG-Vulnerabilitat-Energetica\data\processed\dataset_final_IA.csv"
df_final.to_csv(ruta_final, index=False)

Original: 8331 files.
🔄 Traduint intervals d'anys a números...
Files vàlides per càlcul: 8331
✅ ÈXIT TOTAL! Tenim 1068 seccions censals úniques.
   Codi_Unic Nom_Districte Nom_Barri  Import_Euros  Any_Mitja_Ponderat  \
0  801901001  Ciutat Vella  el Raval         15940                1940   
1  801901002  Ciutat Vella  el Raval         13841                1923   
2  801901003  Ciutat Vella  el Raval         12732                1939   
3  801901004  Ciutat Vella  el Raval         15749                1940   
4  801901005  Ciutat Vella  el Raval         13190                1919   

   Total_Edificis_Zona  
0                   84  
1                   97  
2                   73  
3                  164  
4                  196  
